Replay mode uses recorded fixtures and does not call a live model. Set `NORTHSTAR_MODE=live` with `GEMINI_API_KEY` to opt in, or use `NORTHSTAR_MODE=record` to save synthetic responses.

In [ ]:
from pathlib import Path
from northstar.runtime import get_client, PromptRequest, Message
from lab05 import run_lab, validate_code, render_worksheet

client = get_client(Path("fixtures/replays.json"))  # NORTHSTAR_MODE=replay|live|record

# 05 — Prompt Patterns and Technique Selection

## Scenario

Northstar extracts product codes from support messages. We measure the smallest technique that fixes each observed failure.

## Phase 1: The Zero-Shot Baseline

The recorded baseline includes conversational filler and a wrong number.

In [ ]:
results = run_lab(client)
assert results["zero_accuracy"].numerator == 1
assert results["zero_accuracy"].denominator == 3

## Phase 2: Applying a System Instruction Constraint

A strict output instruction fixes filler but the multiple-number boundary still fails in the recorded run.

In [ ]:
assert results["system_accuracy"].numerator == 2

## Phase 3: Applying Few-Shot Examples for a Decision Boundary

A targeted example distinguishes a product code from an order number.

In [ ]:
assert results["few_accuracy"].numerator == 4
assert results["few_accuracy"].denominator == 4

## Phase 4: when a prompt is the wrong tool

A deterministic regex scores 3/3 on the original suite at zero tokens. It intentionally fails on lowercase `prd 9921`, while the recorded few-shot variant succeeds. The conclusion is: measure, then pick.

In [ ]:
assert results["regex_zero_tokens_accuracy"].numerator == 3
assert validate_code("PRD-9921") == "PRD-9921"
assert validate_code("prd 9921") is None

## Technique selection worksheet

In [ ]:
print(render_worksheet())

## Takeaway

Add one technique at a time, keep evaluation frozen, and retain deterministic code when it solves the measured problem more reliably.

## References

- [Core concepts and workflow](README.md#core-concepts--workflow)
- [Production best practices](README.md#production-best-practices)